## Data paths

Private Colab and Google Drive paths were replaced with repository-relative `data/` paths. Private research data are not included; place permitted local inputs in `data/` or update the path variables for your environment.

# AI Code Detection

## Code-detection workflow

Run Pangram on TXT programming submissions and save one result row per submission. Retry only failed rows, writing repaired results to the same output location without rerunning the full corpus.

This notebook records the code-detection stage conclusion.

### Run Pangram on TXT code submissions

Process the project’s code-submission corpus: each TXT file is one submission and may contain multiple source files. Write one result row per submission for the main code-detection experiment.

In [ ]:
import json
import math
import os
import re
import time
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List

import pandas as pd
import requests
from google.colab import drive

DRIVE_ROOT = os.getenv("DRIVE_ROOT", "data")
SOURCE_FOLDER_NAME = os.getenv("SOURCE_FOLDER_NAME", "KFUPM_Code_Sample")
OUTPUT_ROOT = os.getenv("OUTPUT_ROOT", "pangram_outputs")
PANGRAM_API_KEY = os.getenv("PANGRAM_API_KEY", "")
RUN_DETECTION = os.getenv("RUN_DETECTION", "false").lower() in {"1", "true", "yes"}

API_URL = "https://text.api.pangram.com/v3"
MAX_WORDS_PER_CHUNK = 2500
TIMEOUT = 180
MAX_RETRIES = 5
SECONDS_BETWEEN_REQUESTS = 1.0
BUFFER_MULTIPLIER = 1.10

SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}
KNOWN_CODE_EXTS = {
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".json",
    ".css",
    ".scss",
    ".sass",
    ".less",
    ".html",
    ".htm",
    ".vue",
    ".md",
    ".xml",
    ".yml",
    ".yaml",
    ".graphql",
    ".gql",
    ".sql",
    ".mjs",
    ".cjs",
}


def find_folder_by_name(root: str, folder_name: str) -> Path:
    root_path = Path(root)
    matches = [
        path
        for path in root_path.rglob("*")
        if path.is_dir() and path.name == folder_name
    ]
    if not matches:
        raise FileNotFoundError(
            f"Could not find folder named '{folder_name}' under {root}"
        )
    return sorted(matches, key=lambda path: (len(path.parts), str(path)))[0]


def sanitize_filename(name: str) -> str:
    name = re.sub(r"[^\w.\-]+", "_", name.strip(), flags=re.UNICODE)
    name = re.sub(r"_+", "_", name).strip("._")
    return name or "submission"


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    return re.sub(r"\n{3,}", "\n\n", text).strip()


def word_count(text: str) -> int:
    return len(re.findall(r"\S+", text))


def sanitize_rel_path(rel_path: str) -> str:
    parts = []
    for part in PurePosixPath(rel_path.replace("\\", "/").strip()).parts:
        if part in {"", ".", ".."}:
            continue
        clean = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if clean:
            parts.append(clean)
    cleaned = "/".join(parts)
    return cleaned[:240] if cleaned else "main.js"


def looks_like_path_line(value: str) -> bool:
    value = value.strip()
    if (
        not value
        or value in SEPARATOR_MARKERS
        or value.startswith(("http://", "https://"))
    ):
        return False
    if len(value) > 260 or not any(
        value.lower().endswith(ext) for ext in KNOWN_CODE_EXTS
    ):
        return False

    code_tokens = {
        "import ",
        "from ",
        "const ",
        "let ",
        "var ",
        "function ",
        "class ",
        "<",
        ">",
        "{",
        "}",
        ";",
        "=>",
        "(",
        ")",
        "=",
        "require(",
    }
    return not any(token in value for token in code_tokens) and bool(
        re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", value)
    )


def is_probable_header(lines: List[str], index: int) -> bool:
    if not looks_like_path_line(lines[index].strip()):
        return False
    if index == 0:
        return True

    for previous in range(index - 1, -1, -1):
        value = lines[previous].strip()
        if value:
            return value in SEPARATOR_MARKERS
    return False


def parse_internal_code_files(text: str) -> List[Dict[str, Any]]:
    lines = normalize_text(text).splitlines()
    files: List[Dict[str, Any]] = []
    current_path = None
    current_lines: List[str] = []
    found_header = False
    skip_blank_after_header = False

    def flush_current() -> None:
        nonlocal current_path, current_lines
        if current_path is not None:
            content = "\n".join(current_lines).strip()
            if content:
                files.append(
                    {
                        "path": current_path,
                        "content": content,
                        "word_count": word_count(content),
                    }
                )
        current_path = None
        current_lines = []

    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            continue

        if is_probable_header(lines, index):
            found_header = True
            flush_current()
            current_path = sanitize_rel_path(stripped)
            skip_blank_after_header = True
            continue

        if not found_header:
            current_lines.append(line)
            continue

        if skip_blank_after_header and not stripped:
            skip_blank_after_header = False
            continue

        skip_blank_after_header = False
        if current_path is None:
            current_path = "main.js"
        current_lines.append(line)

    if found_header:
        flush_current()
    else:
        content = "\n".join(lines).strip()
        if content:
            files = [
                {
                    "path": "main.js",
                    "content": content,
                    "word_count": word_count(content),
                }
            ]

    seen: Dict[str, int] = {}
    for file_data in files:
        path = file_data["path"]
        seen[path] = seen.get(path, 0) + 1
        if seen[path] > 1:
            path_obj = Path(path)
            filename = f"{path_obj.stem}__dup{seen[path]}{path_obj.suffix}"
            parent = str(path_obj.parent).replace("\\", "/")
            file_data["path"] = (
                f"{parent}/{filename}" if parent not in {"", "."} else filename
            )

    return files


def split_code_by_line_word_limit(
    text: str, max_words: int = MAX_WORDS_PER_CHUNK
) -> List[str]:
    if word_count(text) <= max_words:
        return [text]

    chunks: List[str] = []
    current_lines: List[str] = []
    current_words = 0

    for line in text.splitlines():
        line_words = word_count(line)
        if line_words > max_words:
            if current_lines:
                chunks.append("\n".join(current_lines).strip())
                current_lines, current_words = [], 0

            tokens = re.findall(r"\S+|\s+", line)
            piece: List[str] = []
            piece_words = 0
            for token in tokens:
                if token.isspace():
                    piece.append(token)
                elif piece_words + 1 > max_words:
                    chunk = "".join(piece).strip()
                    if chunk:
                        chunks.append(chunk)
                    piece, piece_words = [token], 1
                else:
                    piece.append(token)
                    piece_words += 1

            chunk = "".join(piece).strip()
            if chunk:
                chunks.append(chunk)
        elif current_words + line_words <= max_words:
            current_lines.append(line)
            current_words += line_words
        else:
            chunk = "\n".join(current_lines).strip()
            if chunk:
                chunks.append(chunk)
            current_lines, current_words = [line], line_words

    if current_lines:
        chunk = "\n".join(current_lines).strip()
        if chunk:
            chunks.append(chunk)
    return chunks


def call_pangram_v3(text: str, public_dashboard_link: bool = False) -> Dict[str, Any]:
    headers = {
        "Content-Type": "application/json",
        "x-api-key": PANGRAM_API_KEY,
        "Accept": "application/json",
    }
    payload = {"text": text, "public_dashboard_link": public_dashboard_link}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                API_URL, headers=headers, json=payload, timeout=TIMEOUT
            )
            if response.status_code in {429, 500, 502, 503, 504}:
                wait_seconds = min(60, 2**attempt)
                print(
                    f"    Attempt {attempt}: HTTP {response.status_code}; retrying in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue
            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )
            return response.json()
        except Exception as error:
            last_error = error
            if attempt < MAX_RETRIES:
                wait_seconds = min(60, 2**attempt)
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(wait_seconds)

    raise RuntimeError(f"All retries failed. Last error: {last_error}")


def aggregate_chunk_results(
    chunk_results: List[Dict[str, Any]], chunk_word_counts: List[int]
) -> Dict[str, Any]:
    # Weight chunk fractions by submitted code-word count.
    total_weight = sum(max(1, count) for count in chunk_word_counts)
    fractions = {"fraction_ai": 0.0, "fraction_ai_assisted": 0.0, "fraction_human": 0.0}
    segments = {
        "num_ai_segments": 0,
        "num_ai_assisted_segments": 0,
        "num_human_segments": 0,
    }
    dashboard_links, versions, headlines, predictions, prediction_shorts = (
        [],
        [],
        [],
        [],
        [],
    )

    for result, count in zip(chunk_results, chunk_word_counts):
        weight = max(1, count)
        for field in fractions:
            fractions[field] += float(result.get(field, 0) or 0) * weight
        for field in segments:
            segments[field] += int(result.get(field, 0) or 0)
        if result.get("dashboard_link"):
            dashboard_links.append(result["dashboard_link"])
        if result.get("version"):
            versions.append(str(result["version"]))
        if result.get("headline"):
            headlines.append(str(result["headline"]))
        if result.get("prediction"):
            predictions.append(str(result["prediction"]))
        if result.get("prediction_short"):
            prediction_shorts.append(str(result["prediction_short"]))

    if total_weight:
        for field in fractions:
            fractions[field] /= total_weight

    dominant = max(
        [
            ("AI", fractions["fraction_ai"]),
            ("AI-Assisted", fractions["fraction_ai_assisted"]),
            ("Human", fractions["fraction_human"]),
        ],
        key=lambda item: item[1],
    )[0]
    if max(fractions.values()) < 0.60:
        dominant = "Mixed"

    return {
        **fractions,
        "dominant_class": dominant,
        **segments,
        "version": versions[-1] if versions else "",
        "headline_last_chunk": headlines[-1] if headlines else "",
        "prediction_last_chunk": predictions[-1] if predictions else "",
        "prediction_short_last_chunk": (
            prediction_shorts[-1] if prediction_shorts else ""
        ),
        "dashboard_links_joined": " | ".join(dashboard_links),
    }


drive.mount("/content/drive")

output_root = Path(OUTPUT_ROOT)
raw_json_dir = output_root / "raw_json"
output_root.mkdir(parents=True, exist_ok=True)
raw_json_dir.mkdir(parents=True, exist_ok=True)

source_root = find_folder_by_name(DRIVE_ROOT, SOURCE_FOLDER_NAME)
txt_files = sorted(path for path in source_root.rglob("*.txt") if path.is_file())
if not txt_files:
    raise FileNotFoundError(f"No .txt files found under {source_root}")

print("Found source folder:", source_root)
print("Found txt submissions:", len(txt_files))

word_rows = []
total_code_words = 0
total_internal_files = 0

for txt_path in txt_files:
    internal_files = parse_internal_code_files(
        txt_path.read_text(encoding="utf-8", errors="ignore")
    )
    code_words = sum(file_data["word_count"] for file_data in internal_files)
    total_code_words += code_words
    total_internal_files += len(internal_files)
    rel_path = txt_path.relative_to(source_root)

    word_rows.append(
        {
            "source_rel_path": str(rel_path),
            "semester_folder": rel_path.parts[0] if len(rel_path.parts) > 1 else "",
            "file_name": txt_path.name,
            "submission_id": txt_path.stem,
            "internal_file_count": len(internal_files),
            "code_word_count": code_words,
            "estimated_credits_for_this_submission": (
                math.ceil(code_words / 1000) if code_words else 0
            ),
        }
    )

word_df = (
    pd.DataFrame(word_rows)
    .sort_values(["semester_folder", "source_rel_path"])
    .reset_index(drop=True)
)
exact_credits_needed = math.ceil(total_code_words / 1000) if total_code_words else 0
recommended_credits = math.ceil(exact_credits_needed * BUFFER_MULTIPLIER)

word_counts_csv = output_root / "pangram_word_counts.csv"
word_df.to_csv(word_counts_csv, index=False, encoding="utf-8-sig")

word_summary = {
    "source_root": str(source_root),
    "txt_submission_count": len(txt_files),
    "total_internal_code_files": total_internal_files,
    "total_code_words_excluding_headers_and_separators": total_code_words,
    "exact_credits_needed_ceiling_total_words_div_1000": exact_credits_needed,
    "recommended_credits_with_10pct_buffer": recommended_credits,
    "estimated_cost_if_buying_exact_credits_at_0.05_each": round(
        exact_credits_needed * 0.05, 2
    ),
    "estimated_cost_if_buying_recommended_credits_at_0.05_each": round(
        recommended_credits * 0.05, 2
    ),
    "note": "Estimate based on parsed code content, excluding path headers and separator lines.",
}
word_summary_json = output_root / "pangram_word_summary.json"
word_summary_json.write_text(
    json.dumps(word_summary, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("\nWORD COUNT SUMMARY")
print("Total submissions:", len(txt_files))
print("Total internal code files:", total_internal_files)
print("Total code words:", f"{total_code_words:,}")
print("Exact credits needed:", exact_credits_needed)
print("Recommended credits (+10% buffer):", recommended_credits)
print("Saved:", word_counts_csv)
print("Saved:", word_summary_json)

if RUN_DETECTION:
    if not PANGRAM_API_KEY:
        raise ValueError("RUN_DETECTION is enabled but PANGRAM_API_KEY is not set.")

    detection_rows = []
    for index, txt_path in enumerate(txt_files, 1):
        rel_path = txt_path.relative_to(source_root)
        print(f"\n[{index}/{len(txt_files)}] Processing: {rel_path}")

        try:
            internal_files = parse_internal_code_files(
                txt_path.read_text(encoding="utf-8", errors="ignore")
            )
            if not internal_files:
                raise RuntimeError(
                    "No internal code files were parsed from this submission."
                )

            all_results, all_word_counts, internal_file_outputs = [], [], []
            for file_index, file_data in enumerate(internal_files, 1):
                chunks = split_code_by_line_word_limit(file_data["content"])
                file_results, file_word_counts = [], []
                print(
                    f"  Internal file {file_index}/{len(internal_files)}: {file_data['path']} | words={file_data['word_count']:,} | chunks={len(chunks)}"
                )

                for chunk_index, chunk in enumerate(chunks, 1):
                    chunk_count = word_count(chunk)
                    print(
                        f"    Chunk {chunk_index}/{len(chunks)} | words={chunk_count:,}"
                    )
                    result = call_pangram_v3(chunk)
                    file_results.append(result)
                    file_word_counts.append(chunk_count)
                    all_results.append(result)
                    all_word_counts.append(chunk_count)
                    time.sleep(SECONDS_BETWEEN_REQUESTS)

                internal_file_outputs.append(
                    {
                        "path": file_data["path"],
                        "word_count": file_data["word_count"],
                        "chunk_count": len(chunks),
                        "aggregated": aggregate_chunk_results(
                            file_results, file_word_counts
                        ),
                        "raw_responses": file_results,
                    }
                )

            submission_agg = aggregate_chunk_results(all_results, all_word_counts)
            total_words = sum(file_data["word_count"] for file_data in internal_files)
            raw_json_path = (
                raw_json_dir
                / f"{sanitize_filename(str(rel_path.with_suffix('')))}.json"
            )
            raw_payload = {
                "source_rel_path": str(rel_path),
                "internal_file_count": len(internal_files),
                "total_code_word_count": total_words,
                "internal_files": internal_file_outputs,
                "submission_aggregated": submission_agg,
            }
            raw_json_path.write_text(
                json.dumps(raw_payload, ensure_ascii=False, indent=2), encoding="utf-8"
            )

            detection_rows.append(
                {
                    "source_rel_path": str(rel_path),
                    "semester_folder": (
                        rel_path.parts[0] if len(rel_path.parts) > 1 else ""
                    ),
                    "file_name": txt_path.name,
                    "submission_id": txt_path.stem,
                    "internal_file_count": len(internal_files),
                    "code_word_count": total_words,
                    "estimated_credits_for_this_submission": math.ceil(
                        total_words / 1000
                    ),
                    **submission_agg,
                    "raw_json_path": str(raw_json_path),
                    "status": "ok",
                    "error": "",
                }
            )
        except Exception as error:
            print("  FAILED:", error)
            detection_rows.append(
                {
                    "source_rel_path": str(rel_path),
                    "semester_folder": (
                        rel_path.parts[0] if len(rel_path.parts) > 1 else ""
                    ),
                    "file_name": txt_path.name,
                    "submission_id": txt_path.stem,
                    "status": "failed",
                    "error": str(error),
                }
            )

    detection_csv = output_root / "pangram_detection_results.csv"
    pd.DataFrame(detection_rows).to_csv(
        detection_csv, index=False, encoding="utf-8-sig"
    )
    print("\nDETECTION DONE")
    print("Saved:", detection_csv)
    print("Raw JSON folder:", raw_json_dir)
else:
    print(
        "\nDetection skipped. Set PANGRAM_API_KEY and RUN_DETECTION=true to enable it."
    )

### Resume failed Pangram rows

Load the earlier code-detection results, rerun only failed Pangram rows, and overwrite their outputs in the existing results location. This is a recovery step, not a separate experiment.

In [ ]:
import json
import math
import os
import re
import time
from pathlib import Path, PurePosixPath
from typing import Any, Dict, List

import pandas as pd
import requests
from google.colab import drive

MOUNT_PATH = os.environ.get("COLAB_DRIVE_MOUNT", "/content/drive")
DRIVE_ROOT = os.environ.get("DRIVE_ROOT", f"{MOUNT_PATH}/MyDrive")
SOURCE_FOLDER_NAME = os.environ.get("SOURCE_FOLDER_NAME", "KFUPM_Code_Sample")
OUTPUT_ROOT = os.environ.get("PANGRAM_OUTPUT_ROOT", "")
PANGRAM_API_KEY = os.environ.get("PANGRAM_API_KEY", "")
API_URL = "https://text.api.pangram.com/v3"

MAX_WORDS_PER_CHUNK = 2500
TIMEOUT = 180
MAX_RETRIES = 5
SECONDS_BETWEEN_REQUESTS = 1.0

SEPARATOR_MARKERS = {"############", "##########", "==========", "-----", "__________"}
KNOWN_CODE_EXTS = {
    ".js",
    ".jsx",
    ".ts",
    ".tsx",
    ".json",
    ".css",
    ".scss",
    ".sass",
    ".less",
    ".html",
    ".htm",
    ".vue",
    ".md",
    ".xml",
    ".yml",
    ".yaml",
    ".graphql",
    ".gql",
    ".sql",
    ".mjs",
    ".cjs",
}


def find_folder_by_name(root: str, folder_name: str) -> Path:
    root_path = Path(root)
    matches = [
        path
        for path in root_path.rglob("*")
        if path.is_dir() and path.name == folder_name
    ]
    if not matches:
        raise FileNotFoundError(
            f"Could not find folder named '{folder_name}' under {root}"
        )
    matches.sort(key=lambda path: (len(path.parts), str(path)))
    return matches[0]


def sanitize_filename(name: str) -> str:
    name = re.sub(r"[^\w.\-]+", "_", name.strip(), flags=re.UNICODE)
    name = re.sub(r"_+", "_", name).strip("._")
    return name or "submission"


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", "")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def word_count(text: str) -> int:
    return len(re.findall(r"\S+", text))


def sanitize_rel_path(rel_path: str) -> str:
    parts = []
    for part in PurePosixPath(rel_path.replace("\\", "/").strip()).parts:
        if part in ("", ".", ".."):
            continue
        clean = re.sub(r'[<>:"|?*]', "_", part).replace("\x00", "").strip()
        if clean:
            parts.append(clean)
    cleaned = "/".join(parts)
    return cleaned[:240] if cleaned else "main.js"


def looks_like_path_line(text: str) -> bool:
    text = text.strip()
    if (
        not text
        or text in SEPARATOR_MARKERS
        or text.startswith(("http://", "https://"))
    ):
        return False
    if len(text) > 260:
        return False

    lower = text.lower()
    if not any(lower.endswith(extension) for extension in KNOWN_CODE_EXTS):
        return False

    code_tokens = [
        "import ",
        "from ",
        "const ",
        "let ",
        "var ",
        "function ",
        "class ",
        "<",
        ">",
        "{",
        "}",
        ";",
        "=>",
        "(",
        ")",
        "=",
        "require(",
    ]
    if any(token in text for token in code_tokens):
        return False

    return bool(re.fullmatch(r"[A-Za-z0-9_\-./\\ @()\[\]{}+]+", text))


def is_probable_header(lines: List[str], index: int) -> bool:
    if not looks_like_path_line(lines[index].strip()):
        return False

    previous_nonempty = None
    for previous_index in range(index - 1, -1, -1):
        candidate = lines[previous_index].strip()
        if candidate:
            previous_nonempty = candidate
            break

    return index == 0 or previous_nonempty in SEPARATOR_MARKERS


def parse_internal_code_files(text: str) -> List[Dict[str, Any]]:
    lines = normalize_text(text).splitlines()
    files = []
    current_path = None
    current_lines = []
    found_header = False
    skip_one_blank_after_header = False

    def flush_current() -> None:
        nonlocal current_path, current_lines
        if current_path is not None:
            content = "\n".join(current_lines).strip()
            if content:
                files.append(
                    {
                        "path": current_path,
                        "content": content,
                        "word_count": word_count(content),
                    }
                )
        current_path = None
        current_lines = []

    for index, line in enumerate(lines):
        stripped = line.strip()
        if stripped in SEPARATOR_MARKERS:
            continue

        if is_probable_header(lines, index):
            found_header = True
            flush_current()
            current_path = sanitize_rel_path(stripped)
            skip_one_blank_after_header = True
            continue

        if not found_header:
            current_lines.append(line)
            continue

        if skip_one_blank_after_header and not stripped:
            skip_one_blank_after_header = False
            continue

        skip_one_blank_after_header = False
        if current_path is None:
            current_path = "main.js"
        current_lines.append(line)

    if found_header:
        flush_current()
    else:
        content = "\n".join(lines).strip()
        if content:
            files = [
                {
                    "path": "main.js",
                    "content": content,
                    "word_count": word_count(content),
                }
            ]

    seen = {}
    for file_info in files:
        path = file_info["path"]
        seen[path] = seen.get(path, 0) + 1
        if seen[path] > 1:
            path_obj = Path(path)
            name = f"{path_obj.stem}__dup{seen[path]}{path_obj.suffix}"
            parent = str(path_obj.parent).replace("\\", "/")
            file_info["path"] = f"{parent}/{name}" if parent not in ("", ".") else name

    return files


def split_code_by_line_word_limit(
    text: str, max_words: int = MAX_WORDS_PER_CHUNK
) -> List[str]:
    if word_count(text) <= max_words:
        return [text]

    chunks = []
    current_lines = []
    current_words = 0

    for line in text.splitlines():
        line_words = word_count(line)

        if line_words > max_words:
            if current_lines:
                chunks.append("\n".join(current_lines).strip())
                current_lines = []
                current_words = 0

            piece = []
            piece_word_count = 0
            for token in re.findall(r"\S+|\s+", line):
                if token.isspace():
                    piece.append(token)
                    continue
                if piece_word_count + 1 > max_words:
                    chunk = "".join(piece).strip()
                    if chunk:
                        chunks.append(chunk)
                    piece = [token]
                    piece_word_count = 1
                else:
                    piece.append(token)
                    piece_word_count += 1

            chunk = "".join(piece).strip()
            if chunk:
                chunks.append(chunk)
            continue

        if current_words + line_words <= max_words:
            current_lines.append(line)
            current_words += line_words
        else:
            chunk = "\n".join(current_lines).strip()
            if chunk:
                chunks.append(chunk)
            current_lines = [line]
            current_words = line_words

    if current_lines:
        chunk = "\n".join(current_lines).strip()
        if chunk:
            chunks.append(chunk)

    return chunks


def call_pangram_v3(text: str, public_dashboard_link: bool = False) -> Dict[str, Any]:
    headers = {
        "Content-Type": "application/json",
        "x-api-key": PANGRAM_API_KEY,
        "Accept": "application/json",
    }
    payload = {"text": text, "public_dashboard_link": public_dashboard_link}
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.post(
                API_URL, headers=headers, json=payload, timeout=TIMEOUT
            )

            # Insufficient credits cannot be resolved by retrying.
            if response.status_code == 401 and "Insufficient credits" in response.text:
                raise RuntimeError(f"HTTP 401: {response.text}")

            if response.status_code in (429, 500, 502, 503, 504):
                wait_seconds = min(60, 2**attempt)
                print(
                    f"    Attempt {attempt}: HTTP {response.status_code}, retrying in {wait_seconds}s"
                )
                time.sleep(wait_seconds)
                continue

            if response.status_code >= 400:
                raise RuntimeError(
                    f"HTTP {response.status_code}: {response.text[:1000]}"
                )

            return response.json()
        except Exception as error:
            last_error = error
            if "Insufficient credits" in str(error):
                raise
            if attempt < MAX_RETRIES:
                wait_seconds = min(60, 2**attempt)
                print(f"    Attempt {attempt} failed: {error}")
                time.sleep(wait_seconds)

    raise RuntimeError(f"All retries failed. Last error: {last_error}")


def aggregate_chunk_results(
    chunk_results: List[Dict[str, Any]], chunk_word_counts: List[int]
) -> Dict[str, Any]:
    total_weight = sum(max(1, count) for count in chunk_word_counts)
    fractions = {"ai": 0.0, "ai_assisted": 0.0, "human": 0.0}
    segment_counts = {"ai": 0, "ai_assisted": 0, "human": 0}
    dashboard_links, versions, headlines, predictions, prediction_shorts = (
        [],
        [],
        [],
        [],
        [],
    )

    for result, count in zip(chunk_results, chunk_word_counts):
        weight = max(1, count)
        fractions["ai"] += float(result.get("fraction_ai", 0) or 0) * weight
        fractions["ai_assisted"] += (
            float(result.get("fraction_ai_assisted", 0) or 0) * weight
        )
        fractions["human"] += float(result.get("fraction_human", 0) or 0) * weight
        segment_counts["ai"] += int(result.get("num_ai_segments", 0) or 0)
        segment_counts["ai_assisted"] += int(
            result.get("num_ai_assisted_segments", 0) or 0
        )
        segment_counts["human"] += int(result.get("num_human_segments", 0) or 0)

        if result.get("dashboard_link"):
            dashboard_links.append(result["dashboard_link"])
        if result.get("version"):
            versions.append(str(result["version"]))
        if result.get("headline"):
            headlines.append(str(result["headline"]))
        if result.get("prediction"):
            predictions.append(str(result["prediction"]))
        if result.get("prediction_short"):
            prediction_shorts.append(str(result["prediction_short"]))

    if total_weight:
        for key in fractions:
            fractions[key] /= total_weight

    dominant = max(
        [
            ("AI", fractions["ai"]),
            ("AI-Assisted", fractions["ai_assisted"]),
            ("Human", fractions["human"]),
        ],
        key=lambda item: item[1],
    )[0]
    if max(fractions.values()) < 0.60:
        dominant = "Mixed"

    return {
        "fraction_ai": fractions["ai"],
        "fraction_ai_assisted": fractions["ai_assisted"],
        "fraction_human": fractions["human"],
        "dominant_class": dominant,
        "num_ai_segments": segment_counts["ai"],
        "num_ai_assisted_segments": segment_counts["ai_assisted"],
        "num_human_segments": segment_counts["human"],
        "version": versions[-1] if versions else "",
        "headline_last_chunk": headlines[-1] if headlines else "",
        "prediction_last_chunk": predictions[-1] if predictions else "",
        "prediction_short_last_chunk": (
            prediction_shorts[-1] if prediction_shorts else ""
        ),
        "dashboard_links_joined": " | ".join(dashboard_links),
    }


def compute_needed_credits_for_submission(raw_text: str) -> Dict[str, Any]:
    internal_files = parse_internal_code_files(raw_text)
    total_request_credits = 0
    total_requests = 0

    for file_info in internal_files:
        for chunk in split_code_by_line_word_limit(file_info["content"]):
            count = word_count(chunk)
            total_request_credits += math.ceil(count / 1000) if count else 0
            total_requests += 1

    return {
        "internal_file_count": len(internal_files),
        "code_word_count": sum(file_info["word_count"] for file_info in internal_files),
        "estimated_credits_for_this_submission": total_request_credits,
        "estimated_request_count": total_requests,
        "internal_files": internal_files,
    }


drive.mount(MOUNT_PATH)

if not PANGRAM_API_KEY:
    raise ValueError("Set PANGRAM_API_KEY in the environment before running this cell.")
if not OUTPUT_ROOT:
    raise ValueError(
        "Set PANGRAM_OUTPUT_ROOT to the directory containing prior results."
    )

output_root = Path(OUTPUT_ROOT)
raw_json_dir = output_root / "raw_json"
output_root.mkdir(parents=True, exist_ok=True)
raw_json_dir.mkdir(parents=True, exist_ok=True)

results_csv = output_root / "pangram_detection_results.csv"
if not results_csv.exists():
    raise FileNotFoundError(f"Could not find existing results file: {results_csv}")

existing_df = pd.read_csv(results_csv, encoding="utf-8-sig")
if not {"source_rel_path", "status"}.issubset(existing_df.columns):
    raise ValueError(
        "Existing pangram_detection_results.csv does not have the expected columns."
    )

source_root = find_folder_by_name(DRIVE_ROOT, SOURCE_FOLDER_NAME)
failed_df = existing_df[
    existing_df["status"].astype(str).str.lower() == "failed"
].copy()
print(
    "No failed rows found. Nothing to resume."
    if failed_df.empty
    else f"Found {len(failed_df)} failed rows to resume."
)

credit_columns = [
    "source_rel_path",
    "internal_file_count",
    "code_word_count",
    "estimated_request_count",
    "estimated_credits_for_this_submission",
]
failed_credit_rows = []
failed_total_credits = 0

for rel_path_str in failed_df["source_rel_path"].tolist():
    raw_text = (source_root / rel_path_str).read_text(encoding="utf-8", errors="ignore")
    info = compute_needed_credits_for_submission(raw_text)
    failed_total_credits += info["estimated_credits_for_this_submission"]
    failed_credit_rows.append(
        {
            "source_rel_path": rel_path_str,
            "internal_file_count": info["internal_file_count"],
            "code_word_count": info["code_word_count"],
            "estimated_request_count": info["estimated_request_count"],
            "estimated_credits_for_this_submission": info[
                "estimated_credits_for_this_submission"
            ],
        }
    )

# Specifying columns preserves the schema when there are no failed rows.
failed_credit_df = (
    pd.DataFrame(failed_credit_rows, columns=credit_columns)
    .sort_values("source_rel_path")
    .reset_index(drop=True)
)
failed_credit_csv = output_root / "pangram_failed_remainder_credit_estimate.csv"
failed_credit_df.to_csv(failed_credit_csv, index=False, encoding="utf-8-sig")

print("\nFAILED REMAINDER CREDIT ESTIMATE")
print("--------------------------------")
print("Failed submissions:", len(failed_credit_df))
print("Estimated credits needed:", failed_total_credits)
print("Saved:", failed_credit_csv)

updated_rows = {}

for index, rel_path_str in enumerate(failed_df["source_rel_path"].tolist(), 1):
    txt_path = source_root / rel_path_str
    print(f"\n[{index}/{len(failed_df)}] Resuming: {rel_path_str}")

    try:
        info = compute_needed_credits_for_submission(
            txt_path.read_text(encoding="utf-8", errors="ignore")
        )
        internal_files = info["internal_files"]
        if not internal_files:
            raise RuntimeError(
                "No internal code files were parsed from this submission."
            )

        all_chunk_results, all_chunk_word_counts, internal_file_outputs = [], [], []
        for file_index, file_info in enumerate(internal_files, 1):
            chunks = split_code_by_line_word_limit(file_info["content"])
            file_chunk_results, file_chunk_word_counts = [], []
            print(
                f"  Internal file {file_index}/{len(internal_files)}: {file_info['path']} | words={file_info['word_count']:,} | chunks={len(chunks)}"
            )

            for chunk_index, chunk in enumerate(chunks, 1):
                chunk_count = word_count(chunk)
                print(f"    Chunk {chunk_index}/{len(chunks)} | words={chunk_count:,}")
                result = call_pangram_v3(chunk, public_dashboard_link=False)
                file_chunk_results.append(result)
                file_chunk_word_counts.append(chunk_count)
                all_chunk_results.append(result)
                all_chunk_word_counts.append(chunk_count)
                time.sleep(SECONDS_BETWEEN_REQUESTS)

            internal_file_outputs.append(
                {
                    "path": file_info["path"],
                    "word_count": file_info["word_count"],
                    "chunk_count": len(chunks),
                    "aggregated": aggregate_chunk_results(
                        file_chunk_results, file_chunk_word_counts
                    ),
                    "raw_responses": file_chunk_results,
                }
            )

        submission_agg = aggregate_chunk_results(
            all_chunk_results, all_chunk_word_counts
        )
        submission_id = Path(rel_path_str).stem
        raw_json_path = raw_json_dir / f"{sanitize_filename(submission_id)}.json"
        raw_payload = {
            "source_rel_path": rel_path_str,
            "txt_path": str(txt_path),
            "internal_file_count": len(internal_files),
            "total_code_word_count": sum(
                file_info["word_count"] for file_info in internal_files
            ),
            "estimated_credits_for_this_submission": info[
                "estimated_credits_for_this_submission"
            ],
            "estimated_request_count": info["estimated_request_count"],
            "internal_files": internal_file_outputs,
            "submission_aggregated": submission_agg,
        }
        with open(raw_json_path, "w", encoding="utf-8") as file_handle:
            json.dump(raw_payload, file_handle, ensure_ascii=False, indent=2)

        updated_rows[rel_path_str] = {
            "source_rel_path": rel_path_str,
            "semester_folder": (
                Path(rel_path_str).parts[0] if len(Path(rel_path_str).parts) > 1 else ""
            ),
            "file_name": Path(rel_path_str).name,
            "submission_id": submission_id,
            "internal_file_count": len(internal_files),
            "code_word_count": sum(
                file_info["word_count"] for file_info in internal_files
            ),
            "estimated_credits_for_this_submission": info[
                "estimated_credits_for_this_submission"
            ],
            **submission_agg,
            "raw_json_path": str(raw_json_path),
            "status": "ok",
            "error": "",
        }
    except Exception as error:
        print("  FAILED:", error)
        updated_rows[rel_path_str] = {
            "source_rel_path": rel_path_str,
            "semester_folder": (
                Path(rel_path_str).parts[0] if len(Path(rel_path_str).parts) > 1 else ""
            ),
            "file_name": Path(rel_path_str).name,
            "submission_id": Path(rel_path_str).stem,
            "internal_file_count": "",
            "code_word_count": "",
            "estimated_credits_for_this_submission": "",
            "fraction_ai": "",
            "fraction_ai_assisted": "",
            "fraction_human": "",
            "dominant_class": "",
            "num_ai_segments": "",
            "num_ai_assisted_segments": "",
            "num_human_segments": "",
            "version": "",
            "headline_last_chunk": "",
            "prediction_last_chunk": "",
            "prediction_short_last_chunk": "",
            "dashboard_links_joined": "",
            "raw_json_path": "",
            "status": "failed",
            "error": str(error),
        }
        if "Insufficient credits" in str(error):
            print("\nStopped because Pangram still does not have enough credits.")
            break

for rel_path_str, new_row in updated_rows.items():
    mask = existing_df["source_rel_path"].astype(str) == str(rel_path_str)
    for column, value in new_row.items():
        existing_df.loc[mask, column] = value

existing_df.to_csv(results_csv, index=False, encoding="utf-8-sig")
resume_summary = {
    "results_csv_updated": str(results_csv),
    "failed_credit_csv": str(failed_credit_csv),
    "failed_remainder_exact_credits_needed": int(failed_total_credits),
    "updated_rows_count": len(updated_rows),
    "remaining_failed_count_after_resume": int(
        (existing_df["status"].astype(str).str.lower() == "failed").sum()
    ),
}
resume_summary_path = output_root / "pangram_resume_summary.json"
with open(resume_summary_path, "w", encoding="utf-8") as file_handle:
    json.dump(resume_summary, file_handle, ensure_ascii=False, indent=2)

print("\nDONE")
print("Updated same CSV:", results_csv)
print("Failed remainder credit estimate:", failed_credit_csv)
print("Resume summary:", resume_summary_path)
print("Remaining failed rows:", resume_summary["remaining_failed_count_after_resume"])

<h2>Conclusion</h2>

<p>
Pangram classified 63.7% of analyzed code content as AI-like. The weighted AI fraction was 68.0% in the Second Semester sample and 57.7% in the First Semester sample.
</p>

<p>
Among submissions, 65.6% had an AI fraction of at least 50%; 43.8% at least 70%; 31.2% at least 80%; and 21.9% at least 90%. Some submissions were predominantly human or mixed. The AI-assisted category was essentially zero across submissions.
</p>

<p>
These are probabilistic detector outputs, not proof of authorship or a final judgment about an individual submission.
</p>